# 现场演示：已验证178-DSP单画面
把本Notebook上传到板卡 traffic_project/camera_quality_v1 后打开，使用新Kernel。最新四类模型、真实178-DSP双层FPGA＋ARM续算、MJPG、640×480、Q80，沿用已经上板通过的处理路径。

每次识别30秒，会自动结束；原程序上限60秒且最多120张识别帧。它只保存抽样图片、张量和运行记录，不生成连续视频。手机负责拍摄装置和屏幕。

## 1. 初始化：每次新Kernel只运行一次
在Jupyter Running页停止其他使用相机或FPGA的Notebook Kernel，结束其他终端测试。已有完整工程PC备份继续保留。再次运行本格会清空内存中的 sessions 清单，SD上已经保存的结果不会被删除。

In [ ]:
import quality_camera as qc
qc.verify()
sessions = []
programs_stopped = True
backup_on_pc_confirmed = True
print("初始化完成。运行第2格，等待前置校验后出现带框画面，再拍手机识别演示。")


## 2. 开始一段识别：每拍一段运行本格
默认30秒。戴盔静态段可把scene改为helmet_static，裸头静态段为bare_head_static；实际场景写入notes。scene只是记录标签，不改变模型识别类别。先在实际拍摄距离用调焦环把原图调清，再固定镜头。

启动会自动加载匹配新模型的178-DSP图，并完成12例CPU＋12例FPGA对照。前置检查完成后才开始识别计时。一次只有一幅带框画面。

In [ ]:
seconds = 30
scene = "bare_head_static"
notes = "现场演示：请按实际戴盔/未戴盔、静态/推行等情况填写"
camera_index = 0

session = qc.run(jpeg_quality=80, output_width=640, seconds=seconds,
    scene=scene, notes=notes, camera_index=camera_index,
    programs_stopped=programs_stopped,
    backup_on_pc_confirmed=backup_on_pc_confirmed)
sessions.append(session)
print("本段结束；已记录段数：", len(sessions))
print("本段数据目录：", session)


## 3. 可选：拍完全部片段后，统一导出一次
手机视频保存在手机里，和本ZIP不同。ZIP包含板端抽样画面、检测结果、张量和运行记录，供回看、报告及问题分析。可以回到室内后再导出；若已重启Kernel，sessions清单会丢失，但原结果目录仍在SD中。

In [ ]:
if not sessions:
    raise ValueError("当前Kernel没有记录片段；先运行第2格，或使用已保存的RESULT_DIR恢复清单。")
qc.summarize(sessions)
result_zip = qc.export(sessions)
from IPython.display import FileLink, display
import os
display(FileLink(os.path.relpath(result_zip)))
print("请点击上方链接下载ZIP到电脑。手机视频仍在手机相册中。")
